# FedGen — partial-sharing family

## Motivation

### Why partial sharing?
In healthcare FL, sharing the full model θ = [θ_f; θ_p] exposes the feature extractor θ_f,
which learns patient-specific latent representations that can be inverted to reconstruct
sensitive clinical profiles (He et al., 2020). **Partial sharing** keeps θ_f local to each
hospital and only transmits the predictor head θ_p (a single linear layer mapping ℝ^{d} → ℝ^{2}).
This is the minimal information needed for the server to perform knowledge extraction
(Zhu et al. 2021, §3.3) and reduces per-round communication by ~99% vs full-model FedAvg.

### Why FedGen over FedAvg?
Under Dirichlet-partitioned non-IID data (α ∈ {0.5, 1.0, 5.0, 10.0}), FedAvg suffers from
**client drift**: locally optimal models diverge in weight space, so element-wise averaging
produces a poor global model (Li et al. 2020). FedGen addresses this by learning a lightweight
generator G_ω on the server that produces synthetic latent vectors Ẑ consistent with the
ensemble of client predictions. These are broadcast to clients as an **inductive bias**,
regularising local training toward a consensus feature distribution (Zhu et al. 2021, Eq. 5).

### Centroid vs medoid prototype anchor
The generator is trained with a prototype constraint λ·‖G_ω(ŷ,ε) − z̄_ŷ‖² that anchors
generated representations to a per-class representative point z̄_ŷ. Two choices:

- **Centroid** (arithmetic mean of client latent vectors): standard, but on non-convex latent
  manifolds — confirmed by the horseshoe geometry in our latent-space analysis — the centroid
  can fall in empty space between clusters.
- **Medoid** (the actual training point nearest to the centroid): guaranteed to be a real
  patient encoding, lying in a populated region of the manifold. This is relevant for
  **clinical validation**: the generator's anchor is always traceable to a concrete patient
  representation, not an artefact of averaging.

### KD loss: hard CE vs KL ensemble
Two local distillation strategies are compared:

- **Hard CE** (our simplified baseline): L_kd = CE(g_r(Ẑ), ŷ). The synthetic label ŷ is
  treated as ground truth — the local predictor is pushed toward 100% confidence regardless
  of inter-client disagreement.
- **KL ensemble** (Zhu et al. 2021, Eq. 5): L_kd = KL(p_ens ‖ p_local), where
  p_ens = mean_k softmax(W_k·Ẑ + b_k) is a soft distribution. The local predictor learns
  to match the **ensemble's uncertainty**, preserving richer information about decision
  boundaries across heterogeneous clients.

---

## Variant catalogue

### 1. `fedgen_partial` — hard CE + centroid

```
for round t:
  server  → clients : g_r weights, generator ω
  each client k:
    load global g_r  (g_f stays local)
    sample ŷ ~ U{0,1},  ε ~ N(0,I)
    Ẑ = G_ω(ŷ, ε)                        ← frozen, no grad
    for each local epoch:
      step 1 – full model:  L_real = CE_w(g_r(g_f(x)), y)       # class-weighted
      step 2 – g_r only:    L_kd   = CE(g_r(Ẑ), ŷ)             # hard label
    send g_r state + per-class latent stats {μ_y, σ_y, n_y} to server
  server:
    g_r ← FedAvg(g_r states, sample counts)
    prototypes ← weighted aggregate of client {μ_y, σ_y}        # centroid
    generator update:
      min CE(mean_k softmax(W_k·Ẑ' + b_k), ŷ')
        + λ_proto · ‖Ẑ' − z̄_ŷ'‖²                               # prototype constraint
```

Baseline variant. Communicates only the predictor head (~130 params) + generator (~4k params).


### 2. `fedgen_partial_medoid` — hard CE + medoid

Identical to `fedgen_partial` except:
```
  per-class stats: z̄_y = argmin_{z ∈ Z_y} ‖z − mean(Z_y)‖     # medoid replaces centroid
```
Everything upstream (aggregation, generator update) is unchanged — the medoid is a drop-in
replacement for the centroid in the prototype constraint.


### 3. `fedgen_zhu` — KL ensemble + centroid (paper-faithful)

```
for round t:
  server  → clients : g_r weights, generator ω, {W_k, b_k} from previous round
  each client k:
    load global g_r  (g_f stays local)
    sample ŷ ~ U{0,1},  ε ~ N(0,I)
    Ẑ = G_ω(ŷ, ε)                        ← frozen
    p_ens = mean_k softmax(W_k·Ẑ + b_k)  ← frozen ensemble teacher (soft)
    for each local epoch:
      step 1 – full model:  L_real = CE_w(g_r(g_f(x)), y)
      step 2 – g_r only:    L_kd   = -Σ p_ens · log softmax(g_r(Ẑ))   # KL divergence
    send g_r state + per-class latent stats to server
  server:
    g_r ← FedAvg(g_r states, sample counts)
    ensemble teacher ← {W_k, b_k} from this round's g_r states
    prototypes ← centroid
    generator update:  same as fedgen_partial
```

Key difference: the KD target is the ensemble's **soft distribution**, not a hard label.
When clients disagree on a synthetic point (e.g. p_ens = [0.7, 0.3]), the local predictor
learns that uncertainty rather than being forced to 100% confidence.


### 4. `fedgen_zhu_medoid` — KL ensemble + medoid

Identical to `fedgen_zhu` except the prototype anchor uses the medoid.


### 5. `fedgen_zhu_no_proto` — KL ensemble + no prototype constraint (paper Eq. 4 + 5)

```
for round t:
  server  → clients : g_r weights, generator ω, {W_k, b_k} from previous round
  each client k:
    (identical to fedgen_zhu — KL ensemble KD, feature extractor stays local)
  server:
    g_r ← FedAvg(g_r states, sample counts)
    ensemble teacher ← this round's {W_k, b_k}
    generator update:
      min CE(mean_k softmax(W_k·Ẑ' + b_k), ŷ')     # Eq. 4 only — NO prototype term
```

This is the **strictest reading of the paper**: Zhu et al. Eq. 4 trains the generator using
only the CE on ensemble predictions. The prototype constraint λ·‖Ẑ' − z̄_ŷ'‖² is our addition
to anchor the generator to the empirical latent distribution. Setting λ_proto=0 ablates that
addition, isolating whether the anchor helps or hurts.

If `fedgen_zhu_no_proto ≈ fedgen_zhu`, the prototype constraint is redundant — the CE on
ensemble predictions already constrains G_ω sufficiently.
If `fedgen_zhu_no_proto < fedgen_zhu`, the prototype anchoring provides useful regularisation
that the paper's loss alone does not capture.


### 5. `fedgen_gmm` — GMM sampler ablation

```
for round t:
  server  → clients : g_r weights, GMM params {μ_y, σ_y} per class
  each client k:
    load global g_r  (g_f stays local)
    sample Ẑ, ŷ ~ GMMSampler(n_per_class)   ← balanced batch from N(μ_y, σ_y)
    for each local epoch:
      step 1 – full model:  L_real = CE_w(g_r(g_f(x)), y)
      step 2 – g_r only:    L_kd   = CE(g_r(Ẑ), ŷ)
    send g_r state + per-class latent distribution to server
  server:
    g_r ← FedAvg(g_r states, sample counts)
    GMM update: weighted average of client {μ_y, σ_y, n_y}
```

No neural generator, no prototype constraint. Tests whether the learned G_ω adds anything
beyond a simple Gaussian approximation of the per-class latent distribution.
Communication overhead is negligible: 2 × 2 × d × 4 bytes per client per round.
If `fedgen_gmm ≈ fedgen_partial`, the generator's capacity is not earning its keep.

---

## Experimental design

The five variants span a 2×2 design (KD loss × anchor) plus a generator ablation,
crossed with:
- **Data case**: filtered (with partition constraints) and unfiltered
- **Heterogeneity**: α ∈ {0.1, 0.5, 1.0, 5.0, 10.0}
- **Seeds**: {42, 123, 7} for variance estimation

This lets us decompose the sources of improvement:
- KL vs hard CE  → value of soft ensemble knowledge
- Centroid vs medoid → sensitivity to latent geometry
- Neural G_ω vs GMM → value of learned generator
- Partial vs full (compared with `03b`) → privacy–accuracy Pareto frontier

## Setup

In [1]:
import os, sys, json, glob
import numpy as np
import torch

sys.path.insert(0, '..')
from UC1Utils    import ensure_data
from UC1FLUtils  import (
    MLP, Generator, GMMSampler,
    load_clients, save_result,
    N_CLIENTS, FL_ROUNDS, LOCAL_EPOCHS, PATIENCE, NOISE_DIM,
)
from UC1FedGenPartial import (
    run_fedgen_partial, run_fedgen_partial_medoid,
    run_fedgen_zhu, run_fedgen_zhu_medoid, run_fedgen_zhu_no_proto,
    run_fedgen_gmm,
    run_fedgen_zhu_pyhat, run_fedgen_zhu_pyhat_medoid,
    run_fedgen_paper_partial, run_fedgen_paper_partial_no_proto,
    run_fedgen_zhu_code_partial,
    run_fedgen_paper_partial_medoid,
    run_fedgen_zhu_code_partial_medoid,
)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Paths ──────────────────────────────────────────────────────────────────────
FEDERATED_DIR  = '../federated_data'
FILTERED_DIR   = os.path.join(FEDERATED_DIR, 'filtered')
UNFILTERED_DIR = os.path.join(FEDERATED_DIR, 'unfiltered')
RESULTS_DIR    = 'results'

CENTRALIZED_AUC = 0.658

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('figures',   exist_ok=True)

# ── Shared hyperparameters — loaded from FedAvg tuning ────────────────────────
_hp_path = os.path.join(FEDERATED_DIR, 'fl_hyperparams.json')
if not os.path.exists(_hp_path):
    raise FileNotFoundError(
        f'Hyperparameters not found at {_hp_path}.\n'
        f'Run 02_UC1_Federated.ipynb first.'
    )
with open(_hp_path) as f:
    _hp = json.load(f)

HIDDEN_DIM = _hp['hidden_dim']
DROPOUT    = _hp['dropout']
LR         = _hp['lr']
BATCH_SIZE = _hp['batch_size']

SEEDS       = [42, 123, 7]
ALPHA_SWEEP = [0.1, 0.5, 1.0, 5.0, 10.0]
latent_dim  = HIDDEN_DIM // 4

print(f'hidden_dim={HIDDEN_DIM}  dropout={DROPOUT:.4f}  lr={LR:.6f}  '
      f'batch_size={BATCH_SIZE}  latent_dim={latent_dim}')

/mnt/c/Users/Marc/Documents/Uni/4t/TFG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
hidden_dim=512  dropout=0.1803  lr=0.009296  batch_size=256  latent_dim=128


## Communication cost

All five partial variants transmit only the predictor head (≈400 params)
plus either the generator (≈4k params) or the GMM statistics (~256 bytes).
This is where the partial family earns its keep — orders of magnitude cheaper
than FedAvg full.

In [2]:
_m = MLP(99, HIDDEN_DIM, dropout=DROPOUT)   # 99 = placeholder input_dim
_g = Generator(latent_dim, NOISE_DIM)

n_full      = sum(p.numel() for p in _m.parameters())
n_predictor = sum(p.numel() for p in _m.predictor.parameters())
n_gen       = sum(p.numel() for p in _g.parameters())
n_gmm_bytes = GMMSampler(latent_dim, 'cpu').param_bytes()
del _m, _g

B = 4  # float32 bytes
bytes_fedavg_full    = 2 * n_full                  * B * N_CLIENTS
bytes_fedgen_neural  = (2 * n_predictor + n_gen)   * B * N_CLIENTS
bytes_fedgen_gmm     = (2 * n_predictor) * B * N_CLIENTS + n_gmm_bytes * N_CLIENTS

print(f'{"Component":<25} {"Params":>10} {"KB":>8}')
print('-' * 46)
print(f'{"Full MLP":<25} {n_full:>10,} {n_full*B/1024:>8.1f}')
print(f'{"  Feature extractor":<25} {n_full-n_predictor:>10,} {(n_full-n_predictor)*B/1024:>8.1f}')
print(f'{"  Predictor head":<25} {n_predictor:>10,} {n_predictor*B/1024:>8.1f}')
print(f'{"Generator (neural)":<25} {n_gen:>10,} {n_gen*B/1024:>8.1f}')
print(f'{"GMMSampler":<25} {"(no params)":>10} {n_gmm_bytes/1024:>8.3f}')
print()
print(f'{"Variant":<25} {"MB/round":>10} {"vs FedAvg full":>16}')
print('-' * 54)
for name, b in [
    ('FedAvg full (baseline)', bytes_fedavg_full),
    ('FedGen partial neural', bytes_fedgen_neural),
    ('FedGen partial GMM',    bytes_fedgen_gmm),
]:
    ratio = f'{b/bytes_fedavg_full:.3f}×' if 'baseline' not in name else '(baseline)'
    print(f'{name:<25} {b/1024**2:>10.3f} {ratio:>16}')

Component                     Params       KB
----------------------------------------------
Full MLP                     217,474    849.5
  Feature extractor          217,216    848.5
  Predictor head                 258      1.0
Generator (neural)             4,904     19.2
GMMSampler                (no params)    2.000

Variant                     MB/round   vs FedAvg full
------------------------------------------------------
FedAvg full (baseline)         8.296       (baseline)
FedGen partial neural          0.103           0.012×
FedGen partial GMM             0.020           0.002×


## Run the full 5-variant × 2-case × α-sweep × seed grid

Results land in `results/{data_case}/alpha_{α}/seed_{s}/{variant}.json`.
Set `re_train = True` to force re-run of already-completed cells.

In [3]:
re_train = False

VARIANTS = {
    'fedgen_partial': lambda clients, input_dim, seed: run_fedgen_partial(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_partial_medoid': lambda clients, input_dim, seed: run_fedgen_partial_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu': lambda clients, input_dim, seed: run_fedgen_zhu(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_medoid': lambda clients, input_dim, seed: run_fedgen_zhu_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_gmm': lambda clients, input_dim, seed: run_fedgen_gmm(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE,
    ),
    'fedgen_zhu_no_proto': lambda clients, input_dim, seed: run_fedgen_zhu_no_proto(
            clients, input_dim, seed,
            hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
            n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
            local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_pyhat': lambda clients, input_dim, seed: run_fedgen_zhu_pyhat(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_pyhat_medoid': lambda clients, input_dim, seed: run_fedgen_zhu_pyhat_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_paper_partial': lambda clients, input_dim, seed: run_fedgen_paper_partial(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_paper_partial_no_proto': lambda clients, input_dim, seed: run_fedgen_paper_partial_no_proto(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_code_partial': lambda clients, input_dim, seed: run_fedgen_zhu_code_partial(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
     ),
    'fedgen_paper_partial_medoid': lambda clients, input_dim, seed: run_fedgen_paper_partial_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_code_partial_medoid': lambda clients, input_dim, seed: run_fedgen_zhu_code_partial_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
         n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
         local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
     ),
}

for data_case, data_dir in [('filtered', FILTERED_DIR), ('unfiltered', UNFILTERED_DIR)]:
    print(f'\n{"#"*60}\nData case: {data_case}\n{"#"*60}')

    for alpha in ALPHA_SWEEP:
        for seed in SEEDS:
            r_base = os.path.join(RESULTS_DIR, data_case)
            r_dir  = os.path.join(r_base, f'alpha_{alpha}', f'seed_{seed}')

            try:
                clients = load_clients(alpha, data_dir, N_CLIENTS)
            except (FileNotFoundError, ValueError) as e:
                print(f'  Skipping α={alpha} [{data_case}]: {e}')
                continue

            input_dim = clients[0]['X_train'].shape[1]

            for vname, run_fn in VARIANTS.items():
                out_path = os.path.join(r_dir, f'{vname}.json')
                if os.path.exists(out_path) and not re_train:
                    print(f'  [{data_case}] α={alpha} seed={seed} {vname}: already done, skipping.')
                    continue

                print(f'\n{"="*60}\n{vname}  [{data_case}]  α={alpha}  seed={seed}\n{"="*60}')
                auc, pc, hist, cmb = run_fn(clients, input_dim, seed)
                save_result(r_base, alpha, seed, vname, auc, pc, hist, cmb)

print('\nAll partial-family experiments complete.')


############################################################
Data case: filtered
############################################################
  client_0: 28,959 train  pos_rate=6.8%  n_pos=1977
  client_1: 2,472 train  pos_rate=43.7%  n_pos=1080
  client_2: 22,830 train  pos_rate=5.0%  n_pos=1141
  client_3: 6,799 train  pos_rate=43.2%  n_pos=2939
  client_4: 5,401 train  pos_rate=9.0%  n_pos=488
  [filtered] α=0.1 seed=42 fedgen_partial: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_partial_medoid: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu_medoid: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_gmm: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu_no_proto: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu_pyhat: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu_pyhat_medoid: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_paper_parti

## Results summary (partial family only)

In [4]:
PARTIAL_NAMES = {
    'fedgen_partial', 'fedgen_partial_medoid',
    'fedgen_zhu',     'fedgen_zhu_medoid',
    'fedgen_zhu_no_proto',
    'fedgen_gmm','fedgen_zhu_pyhat', 'fedgen_zhu_pyhat_medoid',
    'fedgen_paper_partial', 'fedgen_paper_partial_no_proto',
    'fedgen_zhu_code_partial',
}

print(f'{"Case":<12} {"Alpha":>6}  {"Seed":>6}  {"Variant":<26}  '
      f'{"Test AUC":>9}  {"Rounds":>7}  {"Final MB":>9}')
print('-' * 88)

for path in sorted(glob.glob(f'{RESULTS_DIR}/*/*/seed_*/*.json')):
    parts = path.replace(RESULTS_DIR + '/', '').replace('.json', '').split('/')
    case, alpha_s, seed_s, variant = parts
    if variant not in PARTIAL_NAMES:
        continue
    r = json.load(open(path))
    final_mb = r['cumul_mb'][-1] if r['cumul_mb'] else float('nan')
    print(f'{case:<12} {alpha_s.replace("alpha_",""):>6}  '
          f'{seed_s.replace("seed_",""):>6}  '
          f'{variant:<26}  '
          f'{r["test_auc"]:>9.4f}  '
          f'{len(r["history"]):>7}  '
          f'{final_mb:>9.2f}')

Case          Alpha    Seed  Variant                      Test AUC   Rounds   Final MB
----------------------------------------------------------------------------------------
filtered        0.1     123  fedgen_gmm                     0.6992       17       0.33
filtered        0.1     123  fedgen_paper_partial           0.6669       15       1.55
filtered        0.1     123  fedgen_paper_partial_no_proto     0.6505       13       1.34
filtered        0.1     123  fedgen_partial                 0.7121        6       0.62
filtered        0.1     123  fedgen_partial_medoid          0.6986        6       0.62
filtered        0.1     123  fedgen_zhu                     0.6485       21       2.17
filtered        0.1     123  fedgen_zhu_code_partial        0.6408       15       1.55
filtered        0.1     123  fedgen_zhu_medoid              0.6375       13       1.34
filtered        0.1     123  fedgen_zhu_no_proto            0.6428       10       1.03
filtered        0.1     123  fedgen_zh

---
## Re-train ONLY the `zhu_code` ablations (partial)

The adversarial student term in the `zhu_code` generators was a no-op
(`Z_gen.detach()` killed the gradient); this has been fixed in `UC1FLUtils.py`, so
the term is now active. This cell **re-trains only the `zhu_code` variants** and
**overwrites their stale JSONs** — every other variant is left untouched.

Prerequisite: run the imports/config cell at the top first (it defines the run
functions, hyperparameters, paths, `SEEDS`, `ALPHA_SWEEP`). Then run the cell below;
the validation cell after it sanity-checks that the now-active adversarial term did
not destabilise training.

In [5]:
# RE-TRAIN zhu_code (partial) — OVERWRITES the stale JSONs; other variants untouched.
ZHU_CODE_VARIANTS = {
    'fedgen_zhu_code_partial': run_fedgen_zhu_code_partial,
    'fedgen_zhu_code_partial_medoid': run_fedgen_zhu_code_partial_medoid,
}
_kw = dict(hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
           n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
           local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM)

for data_case, data_dir in [('filtered', FILTERED_DIR), ('unfiltered', UNFILTERED_DIR)]:
    print(f'\n{"#"*60}\nRE-TRAIN zhu_code (partial) — {data_case}\n{"#"*60}')
    for alpha in ALPHA_SWEEP:
        for seed in SEEDS:
            r_base = os.path.join(RESULTS_DIR, data_case)
            try:
                clients = load_clients(alpha, data_dir, N_CLIENTS)
            except (FileNotFoundError, ValueError) as e:
                print(f'  Skipping a={alpha} [{data_case}]: {e}'); continue
            input_dim = clients[0]['X_train'].shape[1]
            for vname, run_fn in ZHU_CODE_VARIANTS.items():
                print(f'\n[RE-TRAIN] {vname}  [{data_case}]  a={alpha}  seed={seed}')
                auc, pc, hist, cmb = run_fn(clients, input_dim, seed, **_kw)
                save_result(r_base, alpha, seed, vname, auc, pc, hist, cmb)   # overwrites stale json
                print(f'   -> saved  test_auc={auc:.4f}  rounds={len(hist)}')
print('\nzhu_code (partial) ablations re-trained.')


############################################################
RE-TRAIN zhu_code (partial) — filtered
############################################################
  client_0: 28,959 train  pos_rate=6.8%  n_pos=1977
  client_1: 2,472 train  pos_rate=43.7%  n_pos=1080
  client_2: 22,830 train  pos_rate=5.0%  n_pos=1141
  client_3: 6,799 train  pos_rate=43.2%  n_pos=2939
  client_4: 5,401 train  pos_rate=9.0%  n_pos=488

[RE-TRAIN] fedgen_zhu_code_partial  [filtered]  a=0.1  seed=42
  [fedgen_zhu_code_partial] R01 (warm-up): val=0.5331 test=0.5224 cumul=0.10MB
  [fedgen_zhu_code_partial] R02 (KD): val=0.5992 test=0.5851 cumul=0.21MB
  [fedgen_zhu_code_partial] R03 (KD): val=0.6470 test=0.6344 cumul=0.31MB
  [fedgen_zhu_code_partial] R04 (KD): val=0.6698 test=0.6491 cumul=0.41MB
  [fedgen_zhu_code_partial] R05 (KD): val=0.6937 test=0.6705 cumul=0.52MB
  [fedgen_zhu_code_partial] R06 (KD): val=0.6806 test=0.6545 cumul=0.62MB
  [fedgen_zhu_code_partial] R07 (KD): val=0.7207 test=0.7056 cumul=

In [6]:
# Validate the re-trained zhu_code results (did the now-active adversarial term stay stable?)
import json, glob, numpy as np
print(f"{'variant':<34}{'case':<11}{'a':<6}{'seed':<6}{'test_auc':<10}{'rounds':<8}{'sane?'}")
ok = True
for v in ZHU_CODE_VARIANTS:
    for p in sorted(glob.glob(os.path.join(RESULTS_DIR, '*', 'alpha_*', 'seed_*', v + '.json'))):
        d = json.load(open(p)); parts = p.replace(os.sep, '/').split('/')
        case, a, s = parts[-4], parts[-3].split('_')[1], parts[-2].split('_')[1]
        auc = d['test_auc']; nr = len(d['history'])
        sane = bool(np.isfinite(auc) and 0.45 < auc < 0.85)
        ok &= sane
        print(f"{v:<34}{case:<11}{a:<6}{s:<6}{auc:<10.4f}{nr:<8}{'OK' if sane else '!! CHECK (diverged?)'}")
print('\n' + ('All re-trained zhu_code results look sane.' if ok
               else '!! Some look off -- inspect the per-round val/test logs above for divergence/NaN.'))

variant                           case       a     seed  test_auc  rounds  sane?
fedgen_zhu_code_partial           filtered   0.1   123   0.6523    8       OK
fedgen_zhu_code_partial           filtered   0.1   42    0.6357    12      OK
fedgen_zhu_code_partial           filtered   0.1   7     0.6579    7       OK
fedgen_zhu_code_partial           filtered   0.5   123   0.6243    10      OK
fedgen_zhu_code_partial           filtered   0.5   42    0.6300    9       OK
fedgen_zhu_code_partial           filtered   0.5   7     0.6321    6       OK
fedgen_zhu_code_partial           filtered   1.0   123   0.6227    10      OK
fedgen_zhu_code_partial           filtered   1.0   42    0.6667    6       OK
fedgen_zhu_code_partial           filtered   1.0   7     0.6231    7       OK
fedgen_zhu_code_partial           filtered   10.0  123   0.5952    6       OK
fedgen_zhu_code_partial           filtered   10.0  42    0.5868    9       OK
fedgen_zhu_code_partial           filtered   10.0  7     0.60